# OCSD – Minh họa từng mô-đun trên một cảnh

Dùng để tạo hình cho Chương 3–4: phác thảo và thông tin đối tượng (M1), ứng viên đối tượng (M2), học định danh (M3), ảnh tiền cảnh và quá trình trộn (M4), bản đồ chú ý và kết quả có/không M5, dãy α. Chạy sau khi RUN_ALL đã dựng bộ dữ liệu.

In [ ]:
#@title 1. Cấu hình (chỉ sửa ô này)
REPO_URL   = "https://github.com/bobbibao/ocsd-thesis.git"  #@param {type:"string"}
BRANCH     = "main"                                             #@param {type:"string"}
DRIVE_ROOT = "/content/drive/MyDrive/KLTN_OCSD"                 #@param {type:"string"}
TIER       = "pilot"   #@param ["pilot", "paper", "full"]
BACKBONE   = "sd15"    #@param ["sd15", "sd21"]
MAX_MINUTES = 600      #@param {type:"number"}
HF_TOKEN   = ""        #@param {type:"string"}
# pilot: kiểm tra nhanh (~1-2 giờ GPU); paper: đủ số liệu cho khóa luận; full: cho bài báo (cần thêm GPU).
# Có thể bấm "Run all" nhiều lần: ảnh/độ đo đã có trên Drive sẽ được bỏ qua và chạy tiếp phần còn lại.

In [ ]:
#@title 2. Gắn Google Drive, kiểm tra GPU, lấy mã nguồn mới nhất
import os, subprocess, sys
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(DRIVE_ROOT, exist_ok=True)
print(subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv", shell=True, capture_output=True, text=True).stdout)
CODE = "/content/KLTN_OCSD"
if not REPO_URL:
    # chưa có repo: dùng bản mã nguồn đã tải lên Drive (thư mục DRIVE_ROOT/code)
    CODE = os.path.join(DRIVE_ROOT, "code")
elif os.path.exists(CODE):
    subprocess.run(f"git -C {CODE} fetch -q origin {BRANCH} && git -C {CODE} reset -q --hard origin/{BRANCH}", shell=True, check=True)
else:
    subprocess.run(f"git clone -q -b {BRANCH} {REPO_URL} {CODE}", shell=True, check=True)
print("Mã nguồn:", CODE, subprocess.run(f"git -C {CODE} log -1 --format='%h %s' 2>/dev/null", shell=True, capture_output=True, text=True).stdout)
sys.path.insert(0, CODE)
# drop modules imported by an earlier run in this kernel, so the code just pulled is what runs
for _m in [m for m in sys.modules if m == "ocsd" or m.startswith("ocsd.")]: del sys.modules[_m]
# HF cache on LOCAL disk: Drive cannot store the cache's symlinks (broken files -> "SafetensorError: header too large").
# Models re-download each new session (~2-4 min on Colab).
os.environ["HF_HOME"] = "/content/hf_cache"
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN

In [ ]:
#@title 3. Install libraries (keeps Colab's PyTorch), start the Drive run log
LOGS = os.path.join(DRIVE_ROOT, "results", "logs"); os.makedirs(LOGS, exist_ok=True)
r = subprocess.run(f"pip install -q -r {CODE}/requirements-colab.txt", shell=True, capture_output=True, text=True)
open(os.path.join(LOGS, "pip_install.log"), "w").write(r.stdout + "\n" + r.stderr)
print(r.stdout[-2000:], r.stderr[-3000:])
if r.returncode != 0:
    raise RuntimeError(f"pip install failed; see {LOGS}/pip_install.log")
import importlib, ocsd; importlib.reload(ocsd)
from ocsd import hfcache
hfcache.repair(os.environ["HF_HOME"])   # delete broken model files so they download again
from ocsd import runlog
runlog.start(os.path.join(DRIVE_ROOT, "results"), CODE)   # every print below also goes to results/logs/run_*.log
from ocsd.config import Paths, ExperimentConfig, TIERS
P = Paths(DRIVE_ROOT); P.makedirs()
E = ExperimentConfig(backbone=BACKBONE, tier=TIER)
E.save(os.path.join(P.results, "experiment_config.json"))
print(E)

In [ ]:
#@title Chọn cảnh
from ocsd.data import list_scenes, load_scene
SCENE_ID = ""  #@param {type:"string"}
dirs = list_scenes(P.benchmarks, "quickdraw", count_bins=["3"], complexities=["medium"])
sdir = next((d for d in dirs if d.endswith(SCENE_ID)), dirs[0]) if SCENE_ID else dirs[0]
scene = load_scene(sdir); print(scene.sid, "|", scene.caption); print(scene.relations)
FIG = os.path.join(P.results, "figures", "demo_" + scene.sid); os.makedirs(FIG, exist_ok=True)

In [ ]:
#@title M1: phác thảo, mặt nạ, hộp, quan hệ
import matplotlib.pyplot as plt, numpy as np
fig, ax = plt.subplots(1, 3, figsize=(15, 5))
ax[0].imshow(scene.sketch, cmap="gray"); ax[0].set_title("Phác thảo cảnh")
col = plt.cm.tab10(np.arange(scene.n) % 10)
ov = np.ones(scene.sketch.shape + (3,))
for i, o in enumerate(scene.objects): ov[o.mask] = col[i][:3]
ax[1].imshow(ov); ax[1].imshow(scene.sketch, cmap="gray", alpha=0.5); ax[1].set_title("Mặt nạ m_i")
ax[2].imshow(scene.sketch, cmap="gray")
for i, o in enumerate(scene.objects):
    x0, y0, x1, y1 = o.box
    ax[2].add_patch(plt.Rectangle((x0, y0), x1-x0, y1-y0, fill=False, color=col[i], lw=2))
    ax[2].text(x0, y0-4, f"{i}: {o.phrase}", color=col[i], fontsize=9)
ax[2].set_title("Hộp b_i: " + "; ".join(f"{i}{r}{j}" for i, j, r in scene.relations[:6]), fontsize=8)
for a in ax: a.axis("off")
plt.tight_layout(); plt.savefig(f"{FIG}/m1.png", dpi=200); plt.show()

In [ ]:
#@title Nạp mô hình
import torch
from ocsd.engine import Engine
from ocsd.vision import Vision
from ocsd.config import OCSDConfig
eng = Engine.from_pretrained(E.backbone, "cuda", E.fp16)
vis = Vision("cuda")
cfg = OCSDConfig()

In [ ]:
#@title M2: ứng viên và ứng viên được chọn
from ocsd.method import object_branch
objs = object_branch(eng, vis, scene, cfg, seed=0)
fig, ax = plt.subplots(2, scene.n, figsize=(3.2*scene.n, 6.4), squeeze=False)
for i, r in enumerate(objs):
    ax[0, i].imshow(r.sketch_crop, cmap="gray"); ax[0, i].set_title(scene.objects[i].phrase)
    shown = r.img.copy(); shown[~r.mask] = (0.35*shown[~r.mask] + 165).astype(np.uint8)
    ax[1, i].imshow(shown); ax[1, i].set_title(f"điểm chọn {r.score:.2f}")
for a in ax.flat: a.axis("off")
plt.tight_layout(); plt.savefig(f"{FIG}/m2.png", dpi=200); plt.show()

In [ ]:
#@title M3 + M4: học định danh, ảnh tiền cảnh, dựng cảnh (có / không M5)
from ocsd.method import compose_foreground, learn_identity, Prepared, generate
fg = compose_foreground(scene, objs, cfg)
log = {}
tokens = learn_identity(eng, scene, objs, cfg, seed=0, log=log); print(log)
prep = Prepared(objs, fg, tokens)
from ocsd.methods import OCSD_VARIANTS, run_baseline
res = {"ControlNet": run_baseline("controlnet", eng, vis, scene, 0),
       "OCSD không M5": generate(eng, vis, scene, prep, OCSD_VARIANTS["abl_no_m5"], 0),
       "OCSD": generate(eng, vis, scene, prep, OCSD_VARIANTS["ocsd"], 0)}
fig, ax = plt.subplots(1, 2 + len(res), figsize=(4*(2+len(res)), 4))
ax[0].imshow(scene.sketch, cmap="gray"); ax[0].set_title("Phác thảo")
ax[1].imshow(fg[0]); ax[1].set_title("Ảnh tiền cảnh x_fg (M4)")
for a, (k, v) in zip(ax[2:], res.items()): a.imshow(v); a.set_title(k)
for a in ax: a.axis("off")
plt.suptitle(scene.caption); plt.tight_layout(); plt.savefig(f"{FIG}/m4_m5.png", dpi=200); plt.show()

In [ ]:
#@title Bản đồ chú ý chéo của token định danh (có / không chú ý giới hạn theo vùng)
from ocsd.attention import token_maps
from ocsd.sketch import scene_global_prompt
from ocsd.method import _lat_mask
gp = scene_global_prompt(eng.tokenizer, scene, tokens)
emb = eng.encode([gp.text]); eot = eng.eot_index(gp.text)
ts = eng.set_timesteps(cfg.steps); t = ts[int(len(ts)*(1-cfg.alpha))]
z = eng.q_sample(eng.to_latent(fg[0]), eng.randn(0), t)
lh = cfg.height // 8
om = torch.stack([_lat_mask(o.mask, lh, lh, eng.device) for o in scene.objects])
eng.lora(True)
fig, ax = plt.subplots(2, scene.n, figsize=(3*scene.n, 6), squeeze=False)
for row, bias in enumerate([False, True]):
    eng.ctrl.set_regions(om, [gp.groups.get(f"obj{i}", []) for i in range(scene.n)], gp.groups.get("bg", []))
    eng.ctrl.bias_enabled, eng.ctrl.lambda_t = bias, cfg.lambda0
    eng.ctrl.store, eng.ctrl.store_res = True, (lh // 4) ** 2; eng.ctrl.reset_maps()
    with torch.no_grad(): eng.unet_eps(z, t, emb, cond_flags=[True])
    maps = token_maps(eng.ctrl.aggregated_map(0), [gp.groups[f"id{i}"] for i in range(scene.n)], eot).cpu()
    eng.ctrl.store = False; eng.ctrl.clear_regions()
    for i in range(scene.n):
        ax[row, i].imshow(maps[i], cmap="jet"); ax[row, i].set_title(("có" if bias else "không") + f" M5(a): {scene.objects[i].cls}", fontsize=8); ax[row, i].axis("off")
eng.lora(False)
plt.tight_layout(); plt.savefig(f"{FIG}/attn.png", dpi=200); plt.show()

In [ ]:
#@title Dãy alpha
alphas = [0.4, 0.5, 0.6, 0.7, 0.8, 1.0]
fig, ax = plt.subplots(1, len(alphas), figsize=(3.2*len(alphas), 3.4))
for a, al in zip(ax, alphas):
    a.imshow(generate(eng, vis, scene, prep, OCSD_VARIANTS["ocsd"].replace(alpha=al, use_verify=False), 0)); a.set_title(f"α = {al}"); a.axis("off")
plt.tight_layout(); plt.savefig(f"{FIG}/alpha.png", dpi=200); plt.show()
eng.reset_identity()

In [ ]:
#@title Nhất quán định danh khi đổi nền (dùng lại M1–M3)
import copy
bgs = ["on a city street", "in a green meadow", "on a sandy beach", "on a snowy field"]
tokens = learn_identity(eng, scene, objs, cfg, seed=0); prep = Prepared(objs, fg, tokens)
fig, ax = plt.subplots(1, len(bgs), figsize=(4*len(bgs), 4))
for a, bg in zip(ax, bgs):
    sc2 = copy.copy(scene); sc2.bg = bg
    a.imshow(generate(eng, vis, sc2, prep, OCSD_VARIANTS["ocsd"], 0)); a.set_title(bg); a.axis("off")
plt.tight_layout(); plt.savefig(f"{FIG}/identity_bg.png", dpi=200); plt.show()
eng.reset_identity()